主要实现功能：整理库存数据

In [1]:
import pandas as pd
from functools import reduce

import openpyxl
import warnings


pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

In [2]:
date_parm = '20260518'

In [3]:
fba_path = f'../src_data/FBA库存/FBA库存-自定义导出-{date_parm}.xlsx'

product_path = f'../src_data/产品库存/产品库存-自定义导出-{date_parm}.xlsx'

yuzhan_path = f'../src_data/预占单/调拨单20250325.xlsx'  # 积加上下载的已拣货未出库状态的调拨单

yiyuzhan_path = f'../src_data/预占单/未出库调拨单导出{date_parm}.xlsx'  # 积加上下载的未拣货未出库状态的调拨单

LogisticsMethod_map = pd.read_excel('../src_data/在途货件/预计到达.xlsx', sheet_name = '物流渠道映射')

yuzhan_df = pd.read_excel(yuzhan_path, sheet_name='明细数据', usecols=['SKU', 'MSKU', '调出数量', '调入仓', '单据状态', '拣货状态', '出库状态'])

In [4]:
# usecols_fba = ['仓库', 'SKU', 'MSKU', 
#                 '已发货(数量)', '正在接收(数量)',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
#                 '预留-运营中心转运(数量)', '预留-运营中心正在处理(数量)', 'FBA可售(数量)', '可用库存(数量)',  # 可用库存(数量) = FBA可售(数量) + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量)
#                 '271~330(数量)', '331~365(数量)', '365+(数量)', 
#                 '7天日均', '15天日均', '30天日均']

usecols_pro = ['仓库', '平台站点', 'SKU', 'MSKU',
                '采购量',  # 采购量  = 采购单未交货数量 = 已下单数量
                '在途量',  # 在途量 = 已发货量 + 待入库量 / 在途量 = 本地-在途 （待质检量）
                '可用量', '未处理预占', '已处理预占']  # 可用量 = 良品量 - 已处理预占量（已拣货，未出库的库存）        未处理预占（未拣货未出库的库存）    

product_df = pd.read_excel(io=product_path, usecols=usecols_pro)

In [5]:
product_df['仓库'].unique()

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '元坤海外仓', '棉帽-广州仓', '小仓库旧货-本地仓',
       '其它品类-广州仓', '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓',
       'DE商易海外仓', 'US商易海外仓', 'CN', 'B端本地仓', 'US易速达海外仓', 'UK商易海外仓',
       'TK本地仓', '易速达2(废弃）:US-GA-1', '异常仓', 'CN易速达:易速达美东GA仓',
       '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
       'IT永翔海外仓', '遮阳帽-广州仓', 'FBT临时仓', '供应商成品本地仓'], dtype=object)

In [6]:
product_df.columns

Index(['仓库', 'MSKU', '平台站点', 'SKU', '采购量', '在途量', '未处理预占', '已处理预占', '可用量'], dtype='object')

In [7]:
# 新增一列“已生产未发货”。(采购量)
# 由于'仓库'列值为'供应商成品本地仓'的行，采购量的值代表“已生产未发货”，则出现以下操作。

# 筛选出'仓库'列值为'供应商成品本地仓'的行，并取列'MSKU', '平台站点', 'SKU', '采购量'
product_df_factory = product_df[product_df['仓库'] == '供应商成品本地仓'][['MSKU', '平台站点', 'SKU', '采购量']]
# 重命名新列为你想要的名称
product_df_factory.rename(columns={'采购量': '已生产未发货'}, inplace=True)

# 合并表，得到新列'已生产未发货'
product_df_new = pd.merge(product_df, product_df_factory, on=['MSKU', '平台站点', 'SKU'], how='left')
# '仓库'列不包含"广州仓"的行，'已生产未发货'列赋值为0
product_df_new.loc[~product_df_new['仓库'].astype(str).str.contains('广州仓', na=False), '已生产未发货'] = 0
# '已生产未发货'列空值填充为0
product_df_new['已生产未发货'] = product_df_new['已生产未发货'].fillna(0)

In [8]:
# 合并'在途量'
# 筛选出'仓库'列值为'供应商成品本地仓'的行，并取列'MSKU', '平台站点', 'SKU', '在途量'
product_df_transit = product_df[product_df['仓库'] == '供应商成品本地仓'][['MSKU', '平台站点', 'SKU', '在途量']]
# 重命名新列为你想要的名称
product_df_transit.rename(columns={'在途量': '在途量_工厂'}, inplace=True)

# 合并表，得到新列'在途量_工厂'
product_df = pd.merge(product_df_new, product_df_transit, on=['MSKU', '平台站点', 'SKU'], how='left')

# '仓库'列不包含"广州仓"的行，'在途量'列赋值为0
product_df.loc[~product_df['仓库'].astype(str).str.contains('广州仓', na=False), '在途量_工厂'] = 0

# '在途量'列空值填充为0
product_df['在途量_工厂'] = product_df['在途量_工厂'].fillna(0)

product_df['在途量'] = product_df['在途量_工厂'] + product_df['在途量']

In [9]:
# product_df.to_excel('../src_data/产品库存/产品库存test5.xlsx', index = False)

In [10]:
# 删除 仓库 列值为'供应商成品本地仓'的行（保留不等于'b'的行）
product_df = product_df[product_df['仓库'] != '供应商成品本地仓']
product_df = product_df.drop(columns=['在途量_工厂'])

In [11]:
# product_df.to_excel('../src_data/产品库存/产品库存test6.xlsx', index = False)

In [12]:
# 20251216 新增逻辑，预占总数等于=已处理预占（已拣货且未出库） + 未处理预占（未拣货且未出库）；
# 可用量原本就减掉了“已处理预占”，还需减掉“未处理预占”，这个预占总数都需要归为在途数，所以要在本地库存中扣减。
product_df['可用量'] = product_df['可用量'] - product_df['未处理预占']
product_df['预占总数'] = product_df['已处理预占'] + product_df['未处理预占']
product_df = product_df.drop(columns = ['已处理预占', '未处理预占'])

# product_df.rename(columns={'SKU0001': 'SKU', '采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '待入库量': '本地-待入库量'}, inplace=True)
# product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '待入库量': '本地-待入库量'}, inplace=True)  # todo
product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '平台站点': '站点'}, inplace=True)  # todo

# 这段代码会找到所有“仓库”为 "TK本地仓" 且“站点”为 "共享" 的行，并将其“站点”值改为 "TIKTOK:TK本地仓:TK"
product_df.loc[(product_df['仓库'] == 'TK本地仓') & (product_df['站点'] == '共享'), '站点'] = 'TIKTOK:TK本地仓:TK本地仓'

In [13]:
usecols_fba = ['仓库', 'SKU', 'MSKU', 
                '货件处理中', '货件已发货', '货件正在接收',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
                '预留-运营中心转运', '预留-运营中心正在处理', '本地可售', '可用量',  # 可用量 = 本地可售 + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量) 良品量=FBA本地可售
                '271~330', '331~365', '365+', 
                '平台7天日均', '平台15天日均', '平台30天日均']
fba_df = pd.read_excel(io=fba_path, usecols=usecols_fba)
fba_df['已发货(数量)'] = fba_df['货件处理中'] + fba_df['货件已发货']
fba_df.rename(columns={'平台7天日均': '7天日均', '平台15天日均': '15天日均', '平台30天日均': '30天日均'}, inplace=True)
# fba_df.rename(columns={'货件已发货': '已发货(数量)', '本地可售': 'FBA可售', '可用量': '可用库存','货件正在接收':'正在接收(数量)'}, inplace=True)
fba_df.rename(columns={'本地可售': 'FBA可售', '可用量': '可用库存','货件正在接收':'正在接收(数量)'}, inplace=True)

In [14]:
'''
# usecols_fba = ['仓库', 'SKU', 'MSKU', 
#                 '已发货(数量)', '正在接收(数量)',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
#                 '预留-运营中心转运(数量)', '预留-运营中心正在处理(数量)', 'FBA可售(数量)', '可用库存(数量)',  # 可用库存(数量) = FBA可售(数量) + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量)
#                 '271~330(数量)', '331~365(数量)', '365+(数量)', 
#                 '7天日均', '15天日均', '30天日均']
usecols_fba = ['仓库', 'SKU', 'MSKU', 
                '货件处理中', '货件已发货', '货件正在接收',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
                '预留-运营中心转运', '预留-运营中心正在处理', '本地可售', '可用量',  # 可用量 = 本地可售 + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量) 良品量=FBA本地可售
                '271~330', '331~365', '365+', 
                '平台7天日均', '平台15天日均', '平台30天日均']

usecols_pro = ['仓库', '平台站点', 'SKU', 'MSKU',
                '采购量',  # 采购量  = 采购单未交货数量 = 已下单数量
                '在途量',  # 在途量 = 已发货量 + 待入库量 / 在途量 = 本地-在途 （待质检量）
                '可用量', '未处理预占', '已处理预占']  # 可用量 = 良品量 - 已处理预占量（已拣货，未出库的库存）        未处理预占（未拣货未出库的库存）    


fba_df = pd.read_excel(io=fba_path, usecols=usecols_fba)
fba_df['已发货(数量)'] = fba_df['货件处理中'] + fba_df['货件已发货']
fba_df.rename(columns={'平台7天日均': '7天日均', '平台15天日均': '15天日均', '平台30天日均': '30天日均'}, inplace=True)
# fba_df.rename(columns={'货件已发货': '已发货(数量)', '本地可售': 'FBA可售', '可用量': '可用库存','货件正在接收':'正在接收(数量)'}, inplace=True)
fba_df.rename(columns={'本地可售': 'FBA可售', '可用量': '可用库存','货件正在接收':'正在接收(数量)'}, inplace=True)

product_df = pd.read_excel(io=product_path, usecols=usecols_pro)

# 20251216 新增逻辑，预占总数等于=已处理预占（已拣货且未出库） + 未处理预占（未拣货且未出库）；
# 可用量原本就减掉了“已处理预占”，还需减掉“未处理预占”，这个预占总数都需要归为在途数，所以要在本地库存中扣减。
product_df['可用量'] = product_df['可用量'] - product_df['未处理预占']
product_df['预占总数'] = product_df['已处理预占'] + product_df['未处理预占']
product_df = product_df.drop(columns = ['已处理预占', '未处理预占'])

# product_df.rename(columns={'SKU0001': 'SKU', '采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '待入库量': '本地-待入库量'}, inplace=True)
# product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '待入库量': '本地-待入库量'}, inplace=True)  # todo
product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '平台站点': '站点'}, inplace=True)  # todo

# 这段代码会找到所有“仓库”为 "TK本地仓" 且“站点”为 "共享" 的行，并将其“站点”值改为 "TIKTOK:TK本地仓:TK"
product_df.loc[(product_df['仓库'] == 'TK本地仓') & (product_df['站点'] == '共享'), '站点'] = 'TIKTOK:TK本地仓:TK本地仓'

yuzhan_df = pd.read_excel(yuzhan_path, sheet_name='明细数据', usecols=['SKU', 'MSKU', '调出数量', '调入仓', '单据状态', '拣货状态', '出库状态'])
'''

'\n# usecols_fba = [\'仓库\', \'SKU\', \'MSKU\', \n#                 \'已发货(数量)\', \'正在接收(数量)\',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）\n#                 \'预留-运营中心转运(数量)\', \'预留-运营中心正在处理(数量)\', \'FBA可售(数量)\', \'可用库存(数量)\',  # 可用库存(数量) = FBA可售(数量) + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量)\n#                 \'271~330(数量)\', \'331~365(数量)\', \'365+(数量)\', \n#                 \'7天日均\', \'15天日均\', \'30天日均\']\nusecols_fba = [\'仓库\', \'SKU\', \'MSKU\', \n                \'货件处理中\', \'货件已发货\', \'货件正在接收\',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）\n                \'预留-运营中心转运\', \'预留-运营中心正在处理\', \'本地可售\', \'可用量\',  # 可用量 = 本地可售 + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量) 良品量=FBA本地可售\n                \'271~330\', \'331~365\', \'365+\', \n                \'平台7天日均\', \'平台15天日均\', \'平台30天日均\']\n\nusecols_pro = [\'仓库\', \'平台站点\', \'SKU\', \'MSKU\',\n                \'采购量\',  # 采购量  = 采购单未交货数量 = 已下单数量\n                \'在途量\',  # 在途量 = 已发货量 + 待入库量 / 在途量 = 本地-在途 （待质检量）\n                \'可用量\', \'未处理预占\', \'已处理预占\

In [15]:
# product_df = pd.read_excel(io=product_path, usecols=usecols_pro)
# product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '平台站点': '站点'}, inplace=True)  # todo

In [16]:
# product_df = new_product_df

In [17]:
product_df.columns

Index(['仓库', 'MSKU', '站点', 'SKU', '已下单数量', '本地-在途', '本地', '已生产未发货', '预占总数'], dtype='object')

# 1、预处理FBA库存数据

In [18]:
# 去掉二次销售MSKU和MX站点的数据
fba_process = fba_df[~(fba_df.MSKU.str.contains('amzn') | fba_df.仓库.str.contains('MX_FBA'))]

# 格式化仓库 shop:site
fba_process['仓库'] = fba_process.仓库.str.split('_FBA', expand=True)[0]

# 把EU站点映射成DE
fba_process['仓库'] = fba_process['仓库'].apply(lambda x: x[:-2] + 'DE' if 'EU' in x else x)

# 获取站点
fba_process['站点'] = fba_process.仓库.str[-2:]

# 定义 长期仓储库存
fba_process['长期仓储库存'] = fba_process[['271~330', '331~365', '365+']].sum(axis=1)
fba_process.drop(columns=['271~330', '331~365', '365+'], inplace=True)

fba_process.query('SKU == "SP001-406 Black 40"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存
2065,SEEKWAY:US,SP001-406 Black 40,SP001-406 Black 40,0,2136,42,234,34,1637,1905,20.00,20.53,18.93,2136,US,0
3190,SEEKWAY:CA,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,CA,0
3955,SEEKWAY:JP,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,JP,0
3956,SEEKWAY:JP,SP001-406 Black 40 NEW,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,JP,0
6613,SENWAYZON:DE,SP001EU-406-Black-41,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,DE,0
13414,SENWAYZON:UK,SP001EU-406-Black-40-new,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,UK,0
14195,SEEKWAY:CA,SP001-406 Black New 40,SP001-406 Black 40,0,0,0,40,45,177,262,1.43,1.40,1.10,0,CA,0
18050,SEEKWAY:JP,JPSP001-406 Black 40,SP001-406 Black 40,0,0,0,0,1,26,27,0.43,0.40,0.40,0,JP,0
20568,SEEKWAY:US,Amazon.Found.B081DYW14R,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,US,0


In [19]:
fba_process.columns

Index(['仓库', 'MSKU', 'SKU', '货件处理中', '货件已发货', '正在接收(数量)', '预留-运营中心转运',
       '预留-运营中心正在处理', 'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '已发货(数量)',
       '站点', '长期仓储库存'],
      dtype='object')

In [20]:
fba_process.仓库.unique()

array(['rivbos:US', 'rivbos:CA', 'rivbos:UK', 'rivbos:JP', 'rivbos:DE',
       'SEEKWAY:US', 'SEEKWAY:CA', 'SEEKWAY:JP', 'SIMARI:JP',
       'SENWAYZON:US', 'SENWAYZON:CA', 'SENWAYZON:UK', 'SENWAYZON:DE',
       'SAYOLA:US', 'SAYOLA:CA', 'RIVMOUNT:US', 'RIVMOUNT:CA',
       'SIMARI:US', 'SIMARI:CA', 'SIMARI:UK', 'SIMARI:DE', 'CIKERWEL:US',
       'CIKERWEL:CA'], dtype=object)

In [21]:
fba_process.站点.unique()

array(['US', 'CA', 'UK', 'JP', 'DE'], dtype=object)

# 2、预处理海外仓库存和本地仓库存

In [22]:
# # 新增 站点 列，根据 仓库 列是否包含 "FBA" 来赋值
# product_df['站点'] = product_df['仓库'].apply(lambda x: x if 'FBA' in str(x) else '共享').str.replace('_FBA', '', regex=False)
# # 新增 店铺 列
# product_df['店铺'] = product_df['站点'].str.replace('EU', 'DE', regex=False)
# # 构建 站点
# product_df['站点'] = product_df.站点.apply(lambda x: x.split(':')[-1] if '共享' not in x else x).str.replace('EU', 'DE', regex=False)

In [23]:
product_df.仓库.unique()

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '元坤海外仓', '棉帽-广州仓', '小仓库旧货-本地仓',
       '其它品类-广州仓', '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓',
       'DE商易海外仓', 'US商易海外仓', 'CN', 'B端本地仓', 'US易速达海外仓', 'UK商易海外仓',
       'TK本地仓', '易速达2(废弃）:US-GA-1', '异常仓', 'CN易速达:易速达美东GA仓',
       '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
       'IT永翔海外仓', '遮阳帽-广州仓', 'FBT临时仓'], dtype=object)

In [24]:
'''
# todo：枚举法排除
# 去除TIKTOK数据
product_df = product_df[~product_df['站点'].str.contains('TIKTOK')]
# 去掉TK仓
product_df = product_df[~product_df['仓库'].str.contains('TK')]

# 去除三方仓数据
product_df = product_df[~product_df['仓库'].str.contains('GA')]

# 去除异常仓 todo
product_df = product_df[~product_df['仓库'].str.contains('异常')]
'''
# 取需要的仓
product_df = product_df[product_df['仓库'].str.contains('广州仓|海外仓|易速达美东GA仓|B端|TK本地仓|顺丰SF')]

# 格式化站点 shop:site
product_df['店铺'] = product_df.站点.apply(lambda x: x if x == '共享' else x[len('AMAZON:'):])

# 构建 站点
product_df['站点'] = product_df.站点.apply(lambda x: x.split(':')[-1] if '共享' not in x else x)


In [25]:
product_df.店铺.unique()

array(['共享', 'SENWAYZON:DE', 'SEEKWAY:US', 'SIMARI:UK', 'GLOW-已删除:US',
       'RIVMOUNT:US', 'ROUNDYAP-已删除:US', 'SENWAYZON:UK', 'SIMARI:DE',
       'SEEKWAY:JP', 'CIKERWEL:US', 'SIMARI:JP', 'RIVMOUNT:CA',
       'SEEKWAY:CA', 'Mr_y-已删除:US', 'rivbos:US', 'rivbos:DE', 'rivbos:UK',
       'rivbos:JP', 'WAYFINDING-已删除:US', 'SENWAYZON:US', 'SAYOLA:US',
       'SENWAYZON:CA', 'rivbos:CA', 'SAYOLA:CA', 'WAYFINDING-已删除:CA',
       'SIMARI:CA', 'GLOW-已删除:CA', 'SIMARI-:United_States', 'TK本地仓:TK本地仓',
       'seekwayer:United_States'], dtype=object)

In [26]:
product_df.仓库.unique()

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '元坤海外仓', '棉帽-广州仓', '其它品类-广州仓',
       '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓', 'DE商易海外仓',
       'US商易海外仓', 'B端本地仓', 'US易速达海外仓', 'UK商易海外仓', 'TK本地仓',
       'CN易速达:易速达美东GA仓', '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓',
       '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓', 'IT永翔海外仓', '遮阳帽-广州仓'],
      dtype=object)

In [27]:
product_df.站点.unique()

array(['共享', 'DE', 'US', 'UK', 'JP', 'CA', 'United_States', 'TK本地仓'],
      dtype=object)

In [28]:
product_df.店铺.unique()

array(['共享', 'SENWAYZON:DE', 'SEEKWAY:US', 'SIMARI:UK', 'GLOW-已删除:US',
       'RIVMOUNT:US', 'ROUNDYAP-已删除:US', 'SENWAYZON:UK', 'SIMARI:DE',
       'SEEKWAY:JP', 'CIKERWEL:US', 'SIMARI:JP', 'RIVMOUNT:CA',
       'SEEKWAY:CA', 'Mr_y-已删除:US', 'rivbos:US', 'rivbos:DE', 'rivbos:UK',
       'rivbos:JP', 'WAYFINDING-已删除:US', 'SENWAYZON:US', 'SAYOLA:US',
       'SENWAYZON:CA', 'rivbos:CA', 'SAYOLA:CA', 'WAYFINDING-已删除:CA',
       'SIMARI:CA', 'GLOW-已删除:CA', 'SIMARI-:United_States', 'TK本地仓:TK本地仓',
       'seekwayer:United_States'], dtype=object)

In [29]:
# # 使用正则表达式一次性排除 '海外' 和 'B端'；只包含本地仓（仓库 值只包含本地仓的行）
# pattern = r'^(?!.*海外)(?!.*B端)(?!.*易速达美东GA仓)(?!.*TK本地仓).*'
# local_inv = product_df[pd.Series(product_df['仓库']).str.contains(pattern, regex=True)]
# local_inv.仓库.unique()

In [30]:
# 使用正则表达式一次性排除 '海外'，'易速达美东GA仓'，'顺丰SF' 和 'B端'；只包含本地仓（仓库 值只包含本地仓的行）
pattern = r'^(?!.*顺丰SF)(?!.*海外)(?!.*B端)(?!.*易速达美东GA仓).*'
local_inv = product_df[pd.Series(product_df['仓库']).str.contains(pattern, regex=True)]
local_inv.仓库.unique()

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '棉帽-广州仓', '其它品类-广州仓', 'TK本地仓',
       '遮阳帽-广州仓'], dtype=object)

In [31]:
local_inv

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,已生产未发货,预占总数,店铺
0,水鞋-广州仓,WP002-208 Black 47,共享,WP002-208 Black 47,0,0.0,0,0.0,0,共享
1,水鞋-广州仓,SWS005-501 circular black 40-41,共享,SWS005-501 circular black 40-41,0,0.0,0,0.0,0,共享
2,眼镜-广州仓,RBK004-1 Black Coating Lens,共享,RBK004-1 Black Coating Lens,0,0.0,0,0.0,0,共享
3,水鞋-广州仓,SP001-407 White 39,共享,SP001-407 White 39,0,0.0,0,0.0,0,共享
4,水鞋-广州仓,SP001-407 White 41,共享,SP001-407 White 41,0,0.0,0,0.0,0,共享
...,...,...,...,...,...,...,...,...,...,...
107533,手套-广州仓,JPSG930-Green S,JP,SMRG930-Green S,0,0.0,5,0.0,0,SEEKWAY:JP
107534,手套-广州仓,JPSG930-Red S,JP,SMRG930-Red S,0,0.0,10,0.0,0,SEEKWAY:JP
107535,手套-广州仓,JPSG930-Red M,JP,SMRG930-Red M,0,0.0,10,0.0,0,SEEKWAY:JP
107536,水鞋-广州仓,SP001-407 White 35,US,SP001-407 White 35,0,0.0,0,0.0,0,SEEKWAY:US


In [32]:
# local_inv

In [33]:
local_inv.query('SKU == "SP001-406 Black 40"')

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,已生产未发货,预占总数,店铺
2881,水鞋-广州仓,SP001-406 Black 40,CA,SP001-406 Black 40,0,0.0,0,0.0,0,SEEKWAY:CA
2961,水鞋-广州仓,SP001-406 Black 40,US,SP001-406 Black 40,2744,340.0,32,0.0,0,SEEKWAY:US
3232,水鞋-广州仓,SP001-406 Black 40,共享,SP001-406 Black 40,0,0.0,0,0.0,0,共享
6271,水鞋-广州仓,SP001EU-406-Black-40,DE,SP001-406 Black 40,0,0.0,0,0.0,0,SENWAYZON:DE
9403,水鞋-广州仓,SP001-406 Black 40,JP,SP001-406 Black 40,0,0.0,0,0.0,0,SEEKWAY:JP
9915,水鞋-广州仓,SP001EU-406-Black-40-new,UK,SP001-406 Black 40,0,0.0,0,0.0,0,SENWAYZON:UK
14944,水鞋-广州仓,SP001EU-406-Black-41,DE,SP001-406 Black 40,0,0.0,0,0.0,0,SENWAYZON:DE
15304,水鞋-广州仓,SP001-406 Black New 40,CA,SP001-406 Black 40,536,32.0,264,0.0,0,SEEKWAY:CA
83800,TK本地仓,SP001-406 Black 40,TK本地仓,SP001-406 Black 40,0,0.0,0,0.0,0,TK本地仓:TK本地仓
88306,水鞋-广州仓,JPSP001-406 Black 40,JP,SP001-406 Black 40,82,0.0,78,0.0,0,SEEKWAY:JP


In [34]:
# local_inv.query('SKU == "SP001-406 Black 40"')

In [35]:
# 只包含海外仓（仓库 值只包含海外仓 的行）
sea_inv = product_df[product_df.仓库.str.contains('海外|易速达美东GA仓|顺丰SF')]
sea_inv.仓库.unique()

array(['元坤海外仓', '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓',
       'DE商易海外仓', 'US商易海外仓', 'US易速达海外仓', 'UK商易海外仓', 'CN易速达:易速达美东GA仓',
       '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
       'IT永翔海外仓'], dtype=object)

In [36]:
# # 只包含海外仓（仓库 值只包含海外仓 的行）
# sea_inv = product_df[product_df.仓库.str.contains('海外|易速达美东GA仓|TK本地仓')]
# sea_inv.仓库.unique()

In [37]:
sea_inv

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,已生产未发货,预占总数,店铺
4816,元坤海外仓,WP005-BA102 Gray 40,共享,WP005-BA102 Gray 40,0,0.0,0,0.0,0,共享
4817,元坤海外仓,SA001-504 Black 43,共享,SA001-504 Black 43,0,0.0,0,0.0,0,共享
4818,元坤海外仓,WP005-BA071 Black 43,共享,WP005-BA071 Black 43,0,0.0,0,0.0,0,共享
4819,元坤海外仓,SP001-AB002 Dark Gray 44,US,SP001-AB002 Dark Gray 44,0,0.0,0,0.0,0,SEEKWAY:US
4820,元坤海外仓,SA001-505 Dark blue 44,共享,SA001-505 Dark blue 44,0,0.0,0,0.0,0,共享
...,...,...,...,...,...,...,...,...,...,...
107574,JP永翔海外仓,wp002-208 Black 41,JP,WP002-208 Black 41,0,0.0,0,0.0,0,SEEKWAY:JP
107575,JP永翔海外仓,wp002-208 Black 44,JP,WP002-208 Black 44,0,0.0,0,0.0,0,SEEKWAY:JP
107576,JP永翔海外仓,wp002-209 Gray 40,JP,WP002-209 Gray 40,0,0.0,0,0.0,0,SEEKWAY:JP
107579,JP永翔海外仓,wp002-215 Purplish blue 36,JP,WP002-215 Purplish blue 36,0,0.0,0,0.0,0,SEEKWAY:JP


# 3、预处理预占单据

## ①处理'已拣货'、'未出库'调拨单据(暂时未用到这部分数据——调拨单)

In [38]:
condition = (
    (yuzhan_df['单据状态'] == '审核通过') &
    (yuzhan_df['拣货状态'] == '已拣货') &
    (yuzhan_df['出库状态'] == '未出库')
)

yuzhan_df = yuzhan_df.loc[condition]

# 区分FBA和海外仓预占
condition = yuzhan_df['调入仓'].str.contains('FBA')

# FBA仓预占
yuzhan_fba = yuzhan_df[condition]

# 海外仓预占
yuzhan_sea = yuzhan_df[~condition]


yuzhan_fba['仓库'] = yuzhan_fba['调入仓'].str.split(pat='_', expand=True)[0]
yuzhan_fba['仓库'] = yuzhan_fba['仓库'].apply(lambda x: x.replace('EU', 'DE') if 'EU' in x else x)

yuzhan_fba = yuzhan_fba.pivot_table(index=['仓库','MSKU'], values=['调出数量'], aggfunc=sum).reset_index().rename(columns={'调出数量': 'FBA预占数'})

In [39]:
yuzhan_df

,SKU,MSKU,调出数量,调入仓,单据状态,拣货状态,出库状态
0,RBK004-2 W Blueblue,RBK004-2 W Blueblue-uk,0,rivbos:EU_FBA,审核通过,已拣货,未出库
1,RBK004-2 W Pink,RBK004-2 W Pink-uk,0,rivbos:EU_FBA,审核通过,已拣货,未出库
2,RBK004-2 W Green,RBK004-2 W Green-uk,0,rivbos:EU_FBA,审核通过,已拣货,未出库
3,RBK004-2 W Black,RBK004-2 W Black-uk,0,rivbos:EU_FBA,审核通过,已拣货,未出库
4,RBK004-2 Black Ice Blue Lens,RBK004-2 Black Ice Blue Lens-uk,0,rivbos:EU_FBA,审核通过,已拣货,未出库
5,RBK004-2 Black Ice Green Lens,RBK004-2 Black Ice Green Lens-uk,0,rivbos:EU_FBA,审核通过,已拣货,未出库
6,RBK004-2 Light Pink,RBK004-2 Light Pink-UK,0,rivbos:EU_FBA,审核通过,已拣货,未出库


In [40]:
yuzhan_fba

,仓库,MSKU,FBA预占数
0,rivbos:DE,RBK004-2 Black Ice Blue Lens-uk,0
1,rivbos:DE,RBK004-2 Black Ice Green Lens-uk,0
2,rivbos:DE,RBK004-2 Light Pink-UK,0
3,rivbos:DE,RBK004-2 W Black-uk,0
4,rivbos:DE,RBK004-2 W Blueblue-uk,0
5,rivbos:DE,RBK004-2 W Green-uk,0
6,rivbos:DE,RBK004-2 W Pink-uk,0


In [41]:
yuzhan_fba.columns

Index(['仓库', 'MSKU', 'FBA预占数'], dtype='object')

## ②处理'未出库'调拨单单据

In [42]:
yiyuzhan_df = pd.read_excel(yiyuzhan_path, sheet_name = '明细数据', usecols=['SKU', 'MSKU', '调出仓', '调入仓', '调出数量', '物流方式', '单据状态', '拣货状态', '出库状态'])
# 将具体的渠道 映射为大渠道：快递、空运、海运
yiyuzhan_df = pd.merge(yiyuzhan_df, LogisticsMethod_map, how = 'left', on = '物流方式')
yiyuzhan_df.drop(columns=['物流方式'], inplace=True)
yiyuzhan_df.rename(columns={'大渠道': '物流方式'}, inplace=True)

# 确保每一个调拨单都能匹配上-物流方式
is_nan = yiyuzhan_df['物流方式'].isna()
if is_nan.sum() > 0:
    raise ValueError("缺失对于大渠道，请更新【物流渠道映射】子表")

In [43]:
yiyuzhan_df['物流方式'].unique()

array(['海运'], dtype=object)

In [44]:
condition = (
    (yiyuzhan_df['单据状态'] == '审核通过') &
    # (yiyuzhan_df['拣货状态'] == '未拣货') &
    (yiyuzhan_df['出库状态'] == '未出库')
)

yiyuzhan_df = yiyuzhan_df.loc[condition]

yiyuzhan_df['仓库'] = yiyuzhan_df['调入仓'].str.split(pat='_', expand=True)[0]
yiyuzhan_df['仓库'] = yiyuzhan_df['仓库'].apply(lambda x: x.replace('EU', 'DE') if 'EU' in x else x)

In [45]:
# 在 pivot_table 之后，手动补全缺失的物流方式列，确保每次结果都有这三列'快递', '空运', '海运'
def pivot_with_all_logistics(df):
    pivot = df.pivot_table(
        index=['仓库', 'MSKU'],
        columns='物流方式',
        values='调出数量',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    pivot.columns.name = None
    for col in ['快递', '空运', '海运', 'FBA自提物流']:
        if col not in pivot.columns:
            pivot[col] = 0
    # return pivot[['仓库', 'MSKU', 'FBA自提物流', '快递', '空运', '海运']]
    return pivot[['仓库', 'MSKU', '快递', '空运', '海运', 'FBA自提物流']]
yiyuzhan_pivot = pivot_with_all_logistics(yiyuzhan_df)

In [46]:
# yiyuzhan_pivot.rename(columns={'FBA自提物流': 'FBA自提物流_预占','快递': '快递_预占','空运': '空运_预占','海运': '海运_预占'}, inplace=True)
yiyuzhan_pivot.rename(columns={'快递': '快递_预占','空运': '空运_预占','海运': '海运_预占','FBA自提物流':'FBA自提物流_预占'}, inplace=True)


In [47]:
yiyuzhan_pivot.columns

Index(['仓库', 'MSKU', '快递_预占', '空运_预占', '海运_预占', 'FBA自提物流_预占'], dtype='object')

In [48]:
yiyuzhan_pivot

,仓库,MSKU,快递_预占,空运_预占,海运_预占,FBA自提物流_预占
0,RIVMOUNT:US,SWR001-101 Constellation 38-39,0,0,0,0


# 4、合并数据

In [49]:
local_inv

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,已生产未发货,预占总数,店铺
0,水鞋-广州仓,WP002-208 Black 47,共享,WP002-208 Black 47,0,0.0,0,0.0,0,共享
1,水鞋-广州仓,SWS005-501 circular black 40-41,共享,SWS005-501 circular black 40-41,0,0.0,0,0.0,0,共享
2,眼镜-广州仓,RBK004-1 Black Coating Lens,共享,RBK004-1 Black Coating Lens,0,0.0,0,0.0,0,共享
3,水鞋-广州仓,SP001-407 White 39,共享,SP001-407 White 39,0,0.0,0,0.0,0,共享
4,水鞋-广州仓,SP001-407 White 41,共享,SP001-407 White 41,0,0.0,0,0.0,0,共享
...,...,...,...,...,...,...,...,...,...,...
107533,手套-广州仓,JPSG930-Green S,JP,SMRG930-Green S,0,0.0,5,0.0,0,SEEKWAY:JP
107534,手套-广州仓,JPSG930-Red S,JP,SMRG930-Red S,0,0.0,10,0.0,0,SEEKWAY:JP
107535,手套-广州仓,JPSG930-Red M,JP,SMRG930-Red M,0,0.0,10,0.0,0,SEEKWAY:JP
107536,水鞋-广州仓,SP001-407 White 35,US,SP001-407 White 35,0,0.0,0,0.0,0,SEEKWAY:US


## 4.1 匹配本地锁仓库存和FBA预占数

In [50]:
 # 筛选出一个不包含 “共享” 店铺的数据框，也就是筛选出‘本地锁仓数’
local_inv_lock = local_inv[~local_inv['店铺'].str.contains('共享')]

# 按照 “店铺”、“SKU” 和 “MSKU” 分组，并计算出每个分组 “本地” 列总和的新数据框
local_inv_lock_piv = local_inv_lock.pivot_table(index=['店铺', 'SKU','MSKU'], values='本地', aggfunc='sum').reset_index()

# FBA库存匹配本地锁仓库存（FBA库存为主表）
new_inv = pd.merge(left=fba_process, right=local_inv_lock_piv, left_on=['仓库', 'SKU','MSKU'], right_on=['店铺', 'SKU','MSKU'], how='left')
new_inv.drop(columns=['店铺'], inplace=True)
new_inv.rename(columns={'本地': '本地锁仓数'}, inplace=True)     #'可用量' -> '本地' -> '本地锁仓数'

# 匹配FBA预占数
new_inv = pd.merge(left=new_inv, right=yuzhan_fba, left_on=['仓库', 'MSKU'], right_on=['仓库', 'MSKU'], how='left')

# 匹配FBA未拣货已预占数
new_inv = pd.merge(left=new_inv, right=yiyuzhan_pivot, left_on=['仓库', 'MSKU'], right_on=['仓库', 'MSKU'], how='left')


In [51]:
new_inv

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占
0,rivbos:US,003-1 sky blue,RBK003-1 sky blue,0,0,0,0,0,0,0,0.00,0.33,0.33,0,US,0,94.0,NaN,NaN,NaN,NaN,NaN
1,rivbos:US,003-1 white,RBK003-1 white,0,0,0,7,2,0,9,0.29,0.13,0.07,0,US,0,31.0,NaN,NaN,NaN,NaN,NaN
2,rivbos:US,003-2 Black Ice Green Lens,RBK003-2 Black Ice Green Lens,0,522,0,249,18,862,1129,25.00,28.40,21.03,522,US,0,310.0,NaN,NaN,NaN,NaN,NaN
3,rivbos:US,003-2 Blue,RBK003-2 Blue,0,0,0,4,0,0,4,0.71,0.33,0.17,0,US,0,31.0,NaN,NaN,NaN,NaN,NaN
4,rivbos:US,004-1 Black,RBK004-1 Black,0,0,0,0,0,0,0,0.00,0.00,0.00,0,US,0,0.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22008,RIVMOUNT:US,WPR002-241 Pink 40,NaN,0,0,0,0,0,0,0,NaN,NaN,NaN,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN
22009,RIVMOUNT:US,WPR002-240 Black 41,NaN,0,0,0,0,0,0,0,NaN,NaN,NaN,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN
22010,RIVMOUNT:US,WPR002-240 Black 45,NaN,0,0,0,0,0,0,0,NaN,NaN,NaN,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN
22011,RIVMOUNT:US,WPR002-240 Black 48,NaN,0,0,0,0,0,0,0,NaN,NaN,NaN,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
local_inv_lock_piv

,店铺,SKU,MSKU,本地
0,CIKERWEL:US,832-1 BlackGrey,832-1 Black&Grey,0
1,CIKERWEL:US,KSK10011-Snowflake Blue 22-23,KSK10011-Snowflake Blue 22-23-US,10
2,CIKERWEL:US,KSK10011-Snowflake Blue 24-25,KSK10011-Snowflake Blue 24-25-US,10
3,CIKERWEL:US,KSK10011-Snowflake Blue 26-27,KSK10011-Snowflake Blue 26-27-US,20
4,CIKERWEL:US,KSK10011-Snowflake Blue 28-29,KSK10011-Snowflake Blue 28-29-US,30
...,...,...,...,...
22281,seekwayer:United_States,SMRG902 Pink L,SG902-Pink-L,0
22282,seekwayer:United_States,SMRG902 Pink M,SG902-Pink-M,0
22283,seekwayer:United_States,SMRG902 Pink S,SG902-Pink-S,0
22284,seekwayer:United_States,SMRG902 Pink XL,SG902-Pink-XL,0


In [53]:
local_inv_lock

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,已生产未发货,预占总数,店铺
77,水鞋-广州仓,SK002-794 White 36-37,DE,SK002-794 White 36-37,0,0.0,0,0.0,0,SENWAYZON:DE
78,水鞋-广州仓,K-06 cat 32-33,US,K-06 cat 32-33,0,0.0,0,0.0,0,SEEKWAY:US
79,水鞋-广州仓,SP001-451 Black 36,DE,SP001-451 Black 36,0,0.0,0,0.0,0,SENWAYZON:DE
80,水鞋-广州仓,WP002-216 Purple 39,UK,WP002-216 Purple 39,0,0.0,0,0.0,0,SIMARI:UK
81,水鞋-广州仓,SA002-CB003 White 37,US,SA002-CB003 White 37,0,0.0,0,0.0,0,GLOW-已删除:US
...,...,...,...,...,...,...,...,...,...,...
107533,手套-广州仓,JPSG930-Green S,JP,SMRG930-Green S,0,0.0,5,0.0,0,SEEKWAY:JP
107534,手套-广州仓,JPSG930-Red S,JP,SMRG930-Red S,0,0.0,10,0.0,0,SEEKWAY:JP
107535,手套-广州仓,JPSG930-Red M,JP,SMRG930-Red M,0,0.0,10,0.0,0,SEEKWAY:JP
107536,水鞋-广州仓,SP001-407 White 35,US,SP001-407 White 35,0,0.0,0,0.0,0,SEEKWAY:US


In [54]:
new_inv.head(10)

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占
0,rivbos:US,003-1 sky blue,RBK003-1 sky blue,0,0,0,0,0,0,0,0.00,0.33,0.33,0,US,0,94.0,NaN,NaN,NaN,NaN,NaN
1,rivbos:US,003-1 white,RBK003-1 white,0,0,0,7,2,0,9,0.29,0.13,0.07,0,US,0,31.0,NaN,NaN,NaN,NaN,NaN
2,rivbos:US,003-2 Black Ice Green Lens,RBK003-2 Black Ice Green Lens,0,522,0,249,18,862,1129,25.00,28.40,21.03,522,US,0,310.0,NaN,NaN,NaN,NaN,NaN
3,rivbos:US,003-2 Blue,RBK003-2 Blue,0,0,0,4,0,0,4,0.71,0.33,0.17,0,US,0,31.0,NaN,NaN,NaN,NaN,NaN
4,rivbos:US,004-1 Black,RBK004-1 Black,0,0,0,0,0,0,0,0.00,0.00,0.00,0,US,0,0.0,NaN,NaN,NaN,NaN,NaN
5,rivbos:US,004-1 Blue,RBK004-1 Blue,0,0,0,0,0,0,0,NaN,NaN,NaN,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN
6,rivbos:US,004-2 Black Ice Red Lens,RBK004-2 Black Ice Red Lens,0,423,21,190,25,1120,1335,17.29,15.93,17.00,423,US,0,830.0,NaN,NaN,NaN,NaN,NaN
7,rivbos:US,004-2 Blue&blue,RBK004-2 W Blueblue,0,2520,278,404,67,4522,4993,49.71,44.27,53.40,2520,US,0,1200.0,NaN,NaN,NaN,NaN,NaN
8,rivbos:US,004-826-white,RBK004-2 W White,0,0,0,0,0,0,0,0.00,0.00,0.00,0,US,0,0.0,NaN,NaN,NaN,NaN,NaN
9,rivbos:US,004-Wayfarer Black,RBK004-2 W Black,0,1237,135,375,875,8215,9465,130.29,113.33,116.60,1237,US,0,68.0,NaN,NaN,NaN,NaN,NaN


In [55]:
# local_inv_lock.to_excel('店铺不包含共享数.xlsx',index = False)

In [56]:
# local_inv_lock_piv.to_excel('不包含共享数.xlsx',index = False)

In [57]:
fba_process.columns

Index(['仓库', 'MSKU', 'SKU', '货件处理中', '货件已发货', '正在接收(数量)', '预留-运营中心转运',
       '预留-运营中心正在处理', 'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '已发货(数量)',
       '站点', '长期仓储库存'],
      dtype='object')

In [58]:
local_inv_lock_piv.query('MSKU == "SP001-406 Black 40"')

,店铺,SKU,MSKU,本地
4844,SEEKWAY:CA,SP001-406 Black 40,SP001-406 Black 40,0
5472,SEEKWAY:JP,SP001-406 Black 40,SP001-406 Black 40,0
7590,SEEKWAY:US,SP001-406 Black 40,SP001-406 Black 40,32
17465,TK本地仓:TK本地仓,SP001-406 Black 40,SP001-406 Black 40,0


In [59]:
local_inv_lock_piv

,店铺,SKU,MSKU,本地
0,CIKERWEL:US,832-1 BlackGrey,832-1 Black&Grey,0
1,CIKERWEL:US,KSK10011-Snowflake Blue 22-23,KSK10011-Snowflake Blue 22-23-US,10
2,CIKERWEL:US,KSK10011-Snowflake Blue 24-25,KSK10011-Snowflake Blue 24-25-US,10
3,CIKERWEL:US,KSK10011-Snowflake Blue 26-27,KSK10011-Snowflake Blue 26-27-US,20
4,CIKERWEL:US,KSK10011-Snowflake Blue 28-29,KSK10011-Snowflake Blue 28-29-US,30
...,...,...,...,...
22281,seekwayer:United_States,SMRG902 Pink L,SG902-Pink-L,0
22282,seekwayer:United_States,SMRG902 Pink M,SG902-Pink-M,0
22283,seekwayer:United_States,SMRG902 Pink S,SG902-Pink-S,0
22284,seekwayer:United_States,SMRG902 Pink XL,SG902-Pink-XL,0


In [60]:
new_inv.query('SKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占
14382,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,9,0,32,1,21,54,0.57,0.8,0.57,9,US,0,0.0,NaN,NaN,NaN,NaN,NaN
17329,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,0,NaN,NaN,NaN,0,CA,0,NaN,NaN,NaN,NaN,NaN,NaN
20215,SEEKWAY:US,Amazon.Found.B0CJJL2LK2,SP001-415 Dark Grey 47,0,0,0,0,0,0,0,0.00,0.0,0.00,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
new_inv.query('SKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占
14382,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,9,0,32,1,21,54,0.57,0.8,0.57,9,US,0,0.0,NaN,NaN,NaN,NaN,NaN
17329,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,0,NaN,NaN,NaN,0,CA,0,NaN,NaN,NaN,NaN,NaN,NaN
20215,SEEKWAY:US,Amazon.Found.B0CJJL2LK2,SP001-415 Dark Grey 47,0,0,0,0,0,0,0,0.00,0.0,0.00,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
new_inv.columns

Index(['仓库', 'MSKU', 'SKU', '货件处理中', '货件已发货', '正在接收(数量)', '预留-运营中心转运',
       '预留-运营中心正在处理', 'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '已发货(数量)',
       '站点', '长期仓储库存', '本地锁仓数', 'FBA预占数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占'],
      dtype='object')

In [63]:
new_inv.columns

Index(['仓库', 'MSKU', 'SKU', '货件处理中', '货件已发货', '正在接收(数量)', '预留-运营中心转运',
       '预留-运营中心正在处理', 'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '已发货(数量)',
       '站点', '长期仓储库存', '本地锁仓数', 'FBA预占数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占'],
      dtype='object')

In [64]:
new_inv.query('MSKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占
14382,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,9,0,32,1,21,54,0.57,0.8,0.57,9,US,0,0.0,NaN,NaN,NaN,NaN,NaN
17329,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,0,NaN,NaN,NaN,0,CA,0,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
new_inv.query('MSKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占
14382,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,9,0,32,1,21,54,0.57,0.8,0.57,9,US,0,0.0,NaN,NaN,NaN,NaN,NaN
17329,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,0,NaN,NaN,NaN,0,CA,0,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
local_inv_lock.query('MSKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,站点,SKU,已下单数量,本地-在途,本地,已生产未发货,预占总数,店铺
15022,水鞋-广州仓,SP001-415 Dark Grey 47,US,SP001-415 Dark Grey 47,109,72.0,0,0.0,0,SEEKWAY:US
83911,TK本地仓,SP001-415 Dark Grey 47,TK本地仓,SP001-415 Dark Grey 47,0,0.0,0,0.0,0,TK本地仓:TK本地仓
106510,TK本地仓,SP001-415 Dark Grey 47,US,SP001-415 Dark Grey 47,0,0.0,0,0.0,0,SEEKWAY:US


## 4.2 匹配本地共享数据


In [67]:
# 采购数据，按照 “SKU”分组，并计算出每个分组 '已下单数量', '本地-在途' 列总和的新数据框。'本地-在途'（待质检数）
# todo
procurement_documents = local_inv.groupby(by=['店铺','SKU','MSKU'])[['已下单数量', '本地-在途', '预占总数', '已生产未发货']].sum().reset_index()
# procurement_documents = local_inv.groupby(by=['SKU'])[['已下单数量', '本地-在途']].sum().reset_index()

# 本地-共享
share_local = local_inv[local_inv['店铺'] == '共享'].groupby(by=['SKU'])['本地'].sum().reset_index().rename(columns={'本地': '本地-共享'})

In [68]:
local_inv.columns

Index(['仓库', 'MSKU', '站点', 'SKU', '已下单数量', '本地-在途', '本地', '已生产未发货', '预占总数',
       '店铺'],
      dtype='object')

In [69]:
procurement_documents.query('SKU == "SP001-415 Dark Grey 47"')

,店铺,SKU,MSKU,已下单数量,本地-在途,预占总数,已生产未发货
7693,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,109,72.0,0,0.0
17576,TK本地仓:TK本地仓,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0.0,0,0.0
25866,共享,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0.0,0,0.0


In [70]:
share_local.query('SKU == "SP001-415 Dark Grey 47"')

,SKU,本地-共享
3580,SP001-415 Dark Grey 47,0


In [71]:
procurement_documents.rename(columns={'店铺':'仓库'}, inplace=True)

procurement_documents.columns

Index(['仓库', 'SKU', 'MSKU', '已下单数量', '本地-在途', '预占总数', '已生产未发货'], dtype='object')

In [72]:
procurement_documents.仓库.unique()

array(['CIKERWEL:US', 'GLOW-已删除:CA', 'GLOW-已删除:US', 'Mr_y-已删除:US',
       'RIVMOUNT:CA', 'RIVMOUNT:US', 'ROUNDYAP-已删除:US', 'SAYOLA:CA',
       'SAYOLA:US', 'SEEKWAY:CA', 'SEEKWAY:JP', 'SEEKWAY:US',
       'SENWAYZON:CA', 'SENWAYZON:DE', 'SENWAYZON:UK', 'SENWAYZON:US',
       'SIMARI-:United_States', 'SIMARI:CA', 'SIMARI:DE', 'SIMARI:JP',
       'SIMARI:UK', 'TK本地仓:TK本地仓', 'WAYFINDING-已删除:CA',
       'WAYFINDING-已删除:US', 'rivbos:CA', 'rivbos:DE', 'rivbos:JP',
       'rivbos:UK', 'rivbos:US', 'seekwayer:United_States', '共享'],
      dtype=object)

In [73]:
dfs = [new_inv, share_local]

# 使用 reduce 函数进行左连接
merged_df = reduce(lambda left, right: pd.merge(left, right, on='SKU', how='left'), dfs)

In [74]:
new_inv.columns

Index(['仓库', 'MSKU', 'SKU', '货件处理中', '货件已发货', '正在接收(数量)', '预留-运营中心转运',
       '预留-运营中心正在处理', 'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '已发货(数量)',
       '站点', '长期仓储库存', '本地锁仓数', 'FBA预占数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占'],
      dtype='object')

In [75]:
new_inv.仓库.unique()   # 此数据框的仓库实际是店铺：站点

array(['rivbos:US', 'rivbos:CA', 'rivbos:UK', 'rivbos:JP', 'rivbos:DE',
       'SEEKWAY:US', 'SEEKWAY:CA', 'SEEKWAY:JP', 'SIMARI:JP',
       'SENWAYZON:US', 'SENWAYZON:CA', 'SENWAYZON:UK', 'SENWAYZON:DE',
       'SAYOLA:US', 'SAYOLA:CA', 'RIVMOUNT:US', 'RIVMOUNT:CA',
       'SIMARI:US', 'SIMARI:CA', 'SIMARI:UK', 'SIMARI:DE', 'CIKERWEL:US',
       'CIKERWEL:CA'], dtype=object)

In [76]:
local_inv.店铺.unique() 

array(['共享', 'SENWAYZON:DE', 'SEEKWAY:US', 'SIMARI:UK', 'GLOW-已删除:US',
       'RIVMOUNT:US', 'ROUNDYAP-已删除:US', 'SENWAYZON:UK', 'SIMARI:DE',
       'SEEKWAY:JP', 'CIKERWEL:US', 'SIMARI:JP', 'RIVMOUNT:CA',
       'SEEKWAY:CA', 'Mr_y-已删除:US', 'rivbos:US', 'rivbos:DE', 'rivbos:UK',
       'rivbos:JP', 'SENWAYZON:US', 'SAYOLA:US', 'SENWAYZON:CA',
       'WAYFINDING-已删除:US', 'rivbos:CA', 'SAYOLA:CA', 'WAYFINDING-已删除:CA',
       'SIMARI:CA', 'GLOW-已删除:CA', 'SIMARI-:United_States', 'TK本地仓:TK本地仓',
       'seekwayer:United_States'], dtype=object)

In [77]:
local_inv.仓库.unique() 

array(['水鞋-广州仓', '眼镜-广州仓', '手套-广州仓', '棉帽-广州仓', '其它品类-广州仓', 'TK本地仓',
       '遮阳帽-广州仓'], dtype=object)

In [78]:
procurement_documents.columns

Index(['仓库', 'SKU', 'MSKU', '已下单数量', '本地-在途', '预占总数', '已生产未发货'], dtype='object')

In [79]:
share_local.columns

Index(['SKU', '本地-共享'], dtype='object')

In [80]:
merged_df.query('SKU == "SP001-415 Dark Grey 47"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占,本地-共享
14382,SEEKWAY:US,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,9,0,32,1,21,54,0.57,0.8,0.57,9,US,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
17329,SEEKWAY:CA,SP001-415 Dark Grey 47,SP001-415 Dark Grey 47,0,0,0,0,0,0,0,NaN,NaN,NaN,0,CA,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0
20215,SEEKWAY:US,Amazon.Found.B0CJJL2LK2,SP001-415 Dark Grey 47,0,0,0,0,0,0,0,0.00,0.0,0.00,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [81]:
merged_df = pd.merge(merged_df, procurement_documents, on = ['仓库', 'SKU', 'MSKU'], how = 'left')

In [82]:
merged_df.query('SKU == "SK001-701 Stripe Black 44-45"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占,本地-共享,已下单数量,本地-在途,预占总数,已生产未发货
1534,SEEKWAY:US,SK001-701 Stripe Black 44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0,0.00,0.00,0.00,0,US,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
1962,SEEKWAY:US,SK002-Stripe Black 701 44-45,SK001-701 Stripe Black 44-45,576,4272,286,3603,689,6086,10378,142.00,101.67,70.63,4848,US,0,11360.0,NaN,NaN,NaN,NaN,NaN,0.0,1441.0,280.0,0.0,0.0
2603,SEEKWAY:CA,SK001-701 Stripe Black 44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0,0.00,0.00,0.00,0,CA,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
3082,SEEKWAY:CA,SK002-Stripe Black 701 44-45,SK001-701 Stripe Black 44-45,0,0,0,100,2,387,489,1.57,2.53,1.83,0,CA,0,120.0,NaN,NaN,NaN,NaN,NaN,0.0,985.0,0.0,0.0,0.0
3792,SEEKWAY:JP,SK002-Stripe Black 701 44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0,0.00,0.00,0.00,0,JP,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
5700,SENWAYZON:UK,SK001EU-701-Stripe-Black-44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0,0.00,0.00,0.00,0,UK,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
6281,SENWAYZON:DE,SK001EU-701-Stripe-Black-44-45,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0,0.00,0.00,0.00,0,DE,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
18107,SEEKWAY:US,Amazon.Found.B07ZQFCXPV,SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0,0.00,0.07,0.23,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
22004,SEEKWAY:CA,SK001-701-Black 44-45(CA),SK001-701 Stripe Black 44-45,0,0,0,0,0,0,0,NaN,NaN,NaN,0,CA,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


# 4.3 数据透视本地库存

In [83]:
# product_local = product_df.pivot_table(index='SKU', columns='站点', values=['本地'], aggfunc=sum, fill_value=0).reset_index()
product_local = local_inv.pivot_table(index='SKU', columns='站点', values=['本地'], aggfunc=sum, fill_value=0).reset_index()


piv_df1 = pd.concat(objs=[product_local['SKU'], product_local['本地']], axis=1)
piv_df2 = local_inv.pivot_table(index='SKU', values=['本地-在途'], aggfunc=sum, fill_value=0).reset_index()

In [84]:
piv_df2.columns

Index(['SKU', '本地-在途'], dtype='object')

In [85]:
local_piv = pd.merge(left=piv_df1, right=piv_df2, on='SKU', how='left')

In [86]:
local_piv.query('SKU == "SP001-406 Black 40"')

,SKU,CA,DE,JP,TK本地仓,UK,US,United_States,共享,本地-在途
3936,SP001-406 Black 40,264,0,78,0,0,32,0,0,372.0


In [87]:
local_piv

,SKU,CA,DE,JP,TK本地仓,UK,US,United_States,共享,本地-在途
0,301 Black,0,0,530,0,0,0,0,0,0.0
1,301 Blue,0,0,200,0,0,0,0,0,0.0
2,301 Camel,0,0,0,0,0,0,0,0,0.0
3,301 Dimgray,0,0,270,0,0,0,0,0,0.0
4,301 Gray,0,0,0,0,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...
7601,YB001 Grey 65cm,0,0,0,0,0,0,0,0,0.0
7602,YB001 Grey 75cm,0,0,0,0,0,0,0,0,0.0
7603,YB001 Pink 55cm,0,0,0,0,0,0,0,0,0.0
7604,YB001 Pink 65cm,0,0,0,0,0,0,0,0,0.0


# 5、处理海外仓库存

In [88]:
# 构建海外仓站点映射
site_map = {'元坤海外仓': 'US', '九方欧洲海外仓': 'DE', 'CA仓搜海外仓' : 'CA', 
            'DE延讯海外仓':'DE', 'UK延讯海外仓' : 'UK', 'JP永翔海外仓' : 'JP',
            'DE商易海外仓' : 'DE', 'US商易海外仓' : 'US', 'US易速达海外仓': 'US', 
            'UK商易海外仓' : 'UK', 'CN易速达:易速达美东GA仓':'US', 'IT永翔海外仓':'IT',
            '顺丰SF:美国特拉华S2仓' : 'US', '顺丰SF:美国洛杉矶S5仓' : 'US', '顺丰SF:美国达拉斯C1仓': 'US', 
            '顺丰SF:美国芝加哥C1仓' : 'US'}

In [89]:
sea_inv['仓库'].unique()

array(['元坤海外仓', '九方欧洲海外仓', 'CA仓搜海外仓', 'DE延讯海外仓', 'UK延讯海外仓', 'JP永翔海外仓',
       'DE商易海外仓', 'US商易海外仓', 'US易速达海外仓', 'UK商易海外仓', 'CN易速达:易速达美东GA仓',
       '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
       'IT永翔海外仓'], dtype=object)

In [90]:
len(site_map)

16

In [91]:
if len(site_map) != sea_inv['仓库'].unique().size:
    raise ValueError(f"海外仓数量有变动，请重新维护变量：site_map ")

In [92]:
# 为海外仓库 映射站点
sea_inv['站点'] = sea_inv['仓库'].apply(site_map.get)

In [93]:
# 海外仓库存全部归为共享
temp_df = sea_inv[['站点','SKU', '仓库', '本地']].pivot_table(index=['站点','SKU'], columns=['仓库'], aggfunc=sum).fillna(0).reset_index()
sea_inv = pd.concat(objs=[temp_df['站点'], temp_df['SKU'], temp_df['本地']], axis=1)


In [94]:
# 预处理海外仓-预占数据
yuzhan_sea['站点'] = yuzhan_sea['调入仓'].apply(site_map.get)

yuzhan_sea = yuzhan_sea.pivot_table(index=['站点', 'SKU'], columns=['调入仓'], values='调出数量', aggfunc=sum).reset_index()

# 重命名字段
columns = yuzhan_sea.columns
new_columns = columns[:2].to_list() + [f'{name}_预占' for name in columns[2:]]
yuzhan_sea.columns = new_columns
if yuzhan_sea.size == 0:
    print('本周无海外仓预占数据')

sea_ret = pd.merge(left=sea_inv, right=yuzhan_sea, on=['站点', 'SKU'], how='outer')

# 填充空值
numeric_columns = sea_ret.select_dtypes(include='number').columns
sea_ret[numeric_columns] = sea_ret[numeric_columns].fillna(0)

本周无海外仓预占数据


# 最后的数据处理与输出

In [95]:
merged_df.query('SKU == "SP001-406 Black 40"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占,本地-共享,已下单数量,本地-在途,预占总数,已生产未发货
2062,SEEKWAY:US,SP001-406 Black 40,SP001-406 Black 40,0,2136,42,234,34,1637,1905,20.00,20.53,18.93,2136,US,0,32.0,NaN,NaN,NaN,NaN,NaN,0.0,2744.0,340.0,0.0,0.0
3187,SEEKWAY:CA,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,CA,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
3945,SEEKWAY:JP,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,JP,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
3946,SEEKWAY:JP,SP001-406 Black 40 NEW,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,JP,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
6580,SENWAYZON:DE,SP001EU-406-Black-41,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,DE,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
13212,SENWAYZON:UK,SP001EU-406-Black-40-new,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,UK,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
13990,SEEKWAY:CA,SP001-406 Black New 40,SP001-406 Black 40,0,0,0,40,45,177,262,1.43,1.40,1.10,0,CA,0,264.0,NaN,NaN,NaN,NaN,NaN,0.0,536.0,32.0,0.0,0.0
17838,SEEKWAY:JP,JPSP001-406 Black 40,SP001-406 Black 40,0,0,0,0,1,26,27,0.43,0.40,0.40,0,JP,0,78.0,NaN,NaN,NaN,NaN,NaN,0.0,82.0,0.0,0.0,0.0
20356,SEEKWAY:US,Amazon.Found.B081DYW14R,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [96]:
merged_df.query('SKU == "SP001-406 Black 40"')

,仓库,MSKU,SKU,货件处理中,货件已发货,正在接收(数量),预留-运营中心转运,预留-运营中心正在处理,FBA可售,可用库存,7天日均,15天日均,30天日均,已发货(数量),站点,长期仓储库存,本地锁仓数,FBA预占数,快递_预占,空运_预占,海运_预占,FBA自提物流_预占,本地-共享,已下单数量,本地-在途,预占总数,已生产未发货
2062,SEEKWAY:US,SP001-406 Black 40,SP001-406 Black 40,0,2136,42,234,34,1637,1905,20.00,20.53,18.93,2136,US,0,32.0,NaN,NaN,NaN,NaN,NaN,0.0,2744.0,340.0,0.0,0.0
3187,SEEKWAY:CA,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,CA,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
3945,SEEKWAY:JP,SP001-406 Black 40,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,JP,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
3946,SEEKWAY:JP,SP001-406 Black 40 NEW,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,JP,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
6580,SENWAYZON:DE,SP001EU-406-Black-41,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,DE,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
13212,SENWAYZON:UK,SP001EU-406-Black-40-new,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,UK,0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
13990,SEEKWAY:CA,SP001-406 Black New 40,SP001-406 Black 40,0,0,0,40,45,177,262,1.43,1.40,1.10,0,CA,0,264.0,NaN,NaN,NaN,NaN,NaN,0.0,536.0,32.0,0.0,0.0
17838,SEEKWAY:JP,JPSP001-406 Black 40,SP001-406 Black 40,0,0,0,0,1,26,27,0.43,0.40,0.40,0,JP,0,78.0,NaN,NaN,NaN,NaN,NaN,0.0,82.0,0.0,0.0,0.0
20356,SEEKWAY:US,Amazon.Found.B081DYW14R,SP001-406 Black 40,0,0,0,0,0,0,0,0.00,0.00,0.00,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [97]:
# 填充缺失值
numeric_columns = merged_df.select_dtypes(include='number').columns

merged_df[numeric_columns] = merged_df[numeric_columns].fillna(0)

In [98]:
merged_df['FBA在途'] = merged_df['FBA预占数'] + merged_df['已发货(数量)'] + merged_df['正在接收(数量)']

In [99]:
merged_df.columns

Index(['仓库', 'MSKU', 'SKU', '货件处理中', '货件已发货', '正在接收(数量)', '预留-运营中心转运',
       '预留-运营中心正在处理', 'FBA可售', '可用库存', '7天日均', '15天日均', '30天日均', '已发货(数量)',
       '站点', '长期仓储库存', '本地锁仓数', 'FBA预占数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占', '本地-共享', '已下单数量', '本地-在途', '预占总数', '已生产未发货', 'FBA在途'],
      dtype='object')

In [100]:
# 调整字段输出顺序
# new_order = ['站点', '仓库', 'SKU', 'MSKU', '预占总数', 'FBA自提物流_预占', '快递_预占', '空运_预占', '海运_预占',
#                 'FBA预占数', '已发货(数量)', '正在接收(数量)', 'FBA在途',
#                 '预留-运营中心转运','预留-运营中心正在处理', 'FBA可售', '可用库存', 
#                 '长期仓储库存', 
#                 '本地锁仓数', '本地-共享', '已下单数量', '本地-在途',
#                 '7天日均', '15天日均', '30天日均']
new_order = ['站点', '仓库', 'SKU', 'MSKU', '预占总数', '快递_预占', '空运_预占', '海运_预占','FBA自提物流_预占',
                'FBA预占数', '已发货(数量)', '正在接收(数量)', 'FBA在途',
                '预留-运营中心转运','预留-运营中心正在处理', 'FBA可售', '可用库存', 
                '长期仓储库存', 
                '本地锁仓数', '本地-共享', '已下单数量', '本地-在途', '已生产未发货',
                '7天日均', '15天日均', '30天日均']
merged_df = merged_df[new_order]

merged_df.rename(columns={'可用库存': 'FBA可用库存'}, inplace=True)

In [101]:
# 要将本地库存数-未拣货数（预占数），因为要“未拣货数（预占数）”视为在途数
# merged_df['本地锁仓数'] = merged_df['本地锁仓数'] - merged_df['预占总数']

In [102]:
merged_df.columns

Index(['站点', '仓库', 'SKU', 'MSKU', '预占总数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占', 'FBA预占数', '已发货(数量)', '正在接收(数量)', 'FBA在途', '预留-运营中心转运',
       '预留-运营中心正在处理', 'FBA可售', 'FBA可用库存', '长期仓储库存', '本地锁仓数', '本地-共享', '已下单数量',
       '本地-在途', '已生产未发货', '7天日均', '15天日均', '30天日均'],
      dtype='object')

In [103]:
# 指定要添加颜色的列
# columns_to_color = {'站点': 'FFF07C82', '仓库': 'FFF07C82', 'SKU': 'FFF07C82', 'MSKU': 'FFF07C82', 
#                     '预占总数': 'FFFCD5B4', 'FBA自提物流_预占': 'FFFCD5B4', '快递_预占': 'FFFCD5B4', '空运_预占': 'FFFCD5B4','海运_预占': 'FFFCD5B4', 
#                 'FBA预占数': 'FF40A070', '已发货(数量)': 'FF40A070', '正在接收(数量)': 'FF40A070', 'FBA在途': 'FF40A070',
#                 '预留-运营中心转运': 'FF9ABEAF','预留-运营中心正在处理': 'FF9ABEAF', 'FBA可售': 'FF9ABEAF', 'FBA可用库存': 'FF9ABEAF', 
#                 '长期仓储库存': 'FF9ABEAF', 
#                 '本地锁仓数': 'FFE2D849', '本地-共享': 'FFE2D849', '已下单数量': 'FFE2D849', '本地-在途': 'FFE2D849',
#                 '7天日均': 'FFE16723', '15天日均': 'FFE16723', '30天日均': 'FFE16723'}
columns_to_color = {'站点': 'FFF07C82', '仓库': 'FFF07C82', 'SKU': 'FFF07C82', 'MSKU': 'FFF07C82', 
                    '预占总数': 'FFFCD5B4', '快递_预占': 'FFFCD5B4', '空运_预占': 'FFFCD5B4','海运_预占': 'FFFCD5B4', 'FBA自提物流_预占': 'FFFCD5B4',
                'FBA预占数': 'FF40A070', '已发货(数量)': 'FF40A070', '正在接收(数量)': 'FF40A070', 'FBA在途': 'FF40A070',
                '预留-运营中心转运': 'FF9ABEAF','预留-运营中心正在处理': 'FF9ABEAF', 'FBA可售': 'FF9ABEAF', 'FBA可用库存': 'FF9ABEAF', 
                '长期仓储库存': 'FF9ABEAF', 
                '本地锁仓数': 'FFE2D849', '本地-共享': 'FFE2D849', '已下单数量': 'FFE2D849', '本地-在途': 'FFE2D849','已生产未发货': 'FFE2D849',
                '7天日均': 'FFE16723', '15天日均': 'FFE16723', '30天日均': 'FFE16723'}

# 将 DataFrame 转换为 Excel 格式，并创建一个 Pandas Excel writer
with pd.ExcelWriter(f'../src_data/处理后的库存/库存{date_parm}.xlsx', engine='openpyxl') as writer:
    merged_df.to_excel(writer, sheet_name='库存汇总', index=False)
    local_piv.to_excel(writer, sheet_name='本地仓库存透视', index=False)
    sea_ret.to_excel(writer, sheet_name='海外仓库存', index=False)
    

    # 获取 Pandas Excel writer 中的 workbook 和 worksheet
    workbook = writer.book
    worksheet = writer.sheets['库存汇总']

    # 使用 Styler 对象为指定列的 header 行添加样式
    rows = merged_df.shape[0]
    for column, color in columns_to_color.items():
        col_idx = merged_df.columns.get_loc(column) + 1
        # print(col_idx)
        for index in range(1, rows+2):
            header_cell = worksheet.cell(index, col_idx)
            header_cell.fill = openpyxl.styles.PatternFill(start_color=color, end_color=color, fill_type='solid')

# 打印结果
print("Excel 文件已生成")


Excel 文件已生成
